# Cross-Lingual Memory: independent-model verification (`main_r1`)

A second model from a different family, **CohereLabs/aya-expanse-8b**, re-labels 300 judge decisions and audits all 165 items'
Hindi/Hinglish text against the English source. Settings: **GPU T4 x2**, **Internet on**, secret **`HF_TOKEN`**
(the HF account must have accepted the Aya licence). Input: the output of **`clm-main-r1-4-judge`**.
Run with **Save Version → Save & Run All**.

## 1. Install and check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm==0.30.0 2>&1 | tail -2
!pip uninstall -y -q torchaudio
import os, sys, glob, json, hashlib, subprocess, shutil
r = subprocess.run(["python", "-c", "import torch, vllm; print('GPUs', torch.cuda.device_count(), 'vllm', vllm.__version__); assert torch.cuda.device_count() == 2"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-1500:])
if r.returncode != 0:
    raise RuntimeError("STOP: environment check failed")

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF token loaded:", bool(os.environ["HF_TOKEN"]))
from huggingface_hub import hf_hub_download
try:
    hf_hub_download("CohereLabs/aya-expanse-8b", "config.json", token=os.environ["HF_TOKEN"]); print("Aya access: OK")
except Exception as e:
    raise RuntimeError("STOP: no access to CohereLabs/aya-expanse-8b. Accept its licence on huggingface.co with the HF_TOKEN account. " + str(e)[:300])

## 2. Rebuild the verification inputs from part 4's output

In [ ]:
!mkdir -p /kaggle/working/verify

In [ ]:
%%writefile /kaggle/working/verify/verify_build.py
"""Builds the independent-model verification prompts. Shared by src/make_verify.py (local) and the Kaggle
notebook (which embeds this file verbatim), so both produce byte-identical jsonl files.
"""
import json

LANG_NAME = {"hi": "Hindi written in Devanagari script",
             "hg": "Hinglish (Hindi written in Latin script, mixed with English words, as Indians text)"}
AUDIT_FIELDS = ["memory_turn", "followup_turn", "distractor_sentence", "q1", "q2", "q3"]
KEY_COLS = ["model", "item_id", "store", "query", "qv", "setup"]


def judge_rows(keys, scored, items, prompts):
    """keys: list of [model, item_id, store, query, qv, setup]; scored: DataFrame of scored.csv."""
    idx = scored.set_index(KEY_COLS)
    out = []
    for k in keys:
        r = idx.loc[tuple(k)]
        it = items[k[1]]
        query = it["text"][k[3]][k[4]]
        answerable = bool(r["answerable"])
        if answerable:
            aliases = [a for sl in it["gold"]["slots"] for a in sl["aliases"]]
            msgs = prompts.judge(query, it["gold"]["answer_en"], aliases, r["response"])
        else:
            msgs = prompts.judge_abstention(query, r["response"])
        sm = r["slot_match"]
        out.append(dict(sid=f"J{len(out):03d}", model=k[0], item_id=k[1], store=k[2], query=k[3], qv=k[4], setup=k[5],
                        answerable=answerable, slot_match=None if sm != sm else int(sm), qwen_judge=str(r["judge"]),
                        messages=msgs))
    return out


def audit_prompt(it, lang):
    en, tr = it["text"]["en"], it["text"][lang]
    fields = [f for f in AUDIT_FIELDS if en.get(f) and tr.get(f)]
    pairs = "\n".join(f"[{f}]\n  English: {en[f]}\n  {lang.upper()}: {tr[f]}" for f in fields)
    if it["is_answerable"]:
        key = (f"Key fact that must survive in the translated memory_turn: {it['gold']['answer_en']}\n"
               f"Accepted forms: {'; '.join(a for sl in it['gold']['slots'] for a in sl['aliases'])}\n")
    else:
        key = "This item has only questions (no stored fact); key_fact_present should be \"n/a\".\n"
    user = (
        f"You are auditing a translation for a research dataset. Each field below has an English source and its "
        f"version in {LANG_NAME[lang]}.\n{key}\n{pairs}\n\n"
        "Check:\n"
        "1. meaning_same: every translated field means the same as its English source (no added, dropped or changed facts, "
        "names, numbers or dates).\n"
        "2. key_fact_present: the key fact is stated in the translated memory_turn.\n"
        "3. questions_same: the translated questions ask for the same information as the English ones.\n"
        f"4. natural: it reads like natural {'Hindi' if lang == 'hi' else 'Hinglish'} a native speaker would write.\n"
        "Reply with only this JSON, no other text:\n"
        '{"meaning_same": "yes|no", "key_fact_present": "yes|no|n/a", "questions_same": "yes|no", '
        '"natural": "yes|no", "problems": "<short description, or none>"}'
    )
    return fields, [{"role": "user", "content": user}]


def audit_rows(items):
    out = []
    for iid, it in items.items():
        for lang in ("hi", "hg"):
            fields, msgs = audit_prompt(it, lang)
            out.append(dict(aid=f"{iid}|{lang}", item_id=iid, lang=lang, answerable=it["is_answerable"],
                            fields=fields, messages=msgs))
    return out


def dumps(rows):
    return "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)


In [ ]:
%%writefile /kaggle/working/verify/judge_keys.json
[["llama3.1-8b", "E020", "en", "hg", "q1", "RM-S2"], ["qwen3-8b", "F028", "en", "hg", "q1", "RM-S2"], ["llama3.1-8b", "F015", "hi", "en", "q2", "OR-S3"], ["llama3.1-8b", "F007", "hi", "en", "q2", "RM-S3"], ["qwen3-8b", "F016", "en", "hi", "q2", "OR-S2"], ["qwen3-8b", "E026", "en", "hi", "q2", "OR-S1"], ["llama3.1-8b", "P038", "en", "hi", "q2", "OR-S3"], ["qwen3-8b", "F021", "hg", "hi", "q1", "FC"], ["llama3.1-8b", "F027", "en", "hg", "q1", "OR-S3"], ["qwen3-8b", "E031", "hg", "hg", "q2", "OR-S1"], ["llama3.1-8b", "F024", "hg", "en", "q2", "FC"], ["llama3.1-8b", "F046", "hi", "hi", "q1", "OR-S2"], ["llama3.1-8b", "F048", "hg", "hi", "q1", "OR-S1"], ["qwen3-8b", "E046", "hi", "hi", "q2", "RM-S3"], ["qwen3-8b", "E037", "hi", "en", "q1", "OR-S1"], ["qwen3-8b", "F026", "hg", "hg", "q2", "OR-S2"], ["qwen3-8b", "P039", "hi", "en", "q1", "RM-S3"], ["qwen3-8b", "F014", "hg", "en", "q1", "OR-S3"], ["llama3.1-8b", "E003", "hi", "en", "q1", "FC"], ["qwen3-8b", "E025", "en", "hg", "q1", "FC"], ["llama3.1-8b", "E002", "en", "hg", "q2", "RM-S3"], ["qwen3-8b", "P013", "en", "en", "q1", "FC"], ["llama3.1-8b", "E027", "en", "hi", "q2", "RM-S3"], ["llama3.1-8b", "P039", "hi", "hg", "q2", "OR-S2"], ["llama3.1-8b", "E037", "en", "hi", "q2", "RM-S2"], ["qwen3-8b", "P017", "hi", "hg", "q1", "FC"], ["qwen3-8b", "E032", "en", "en", "q1", "RM-S3"], ["qwen3-8b", "P025", "hg", "en", "q1", "RM-S2"], ["llama3.1-8b", "P044", "en", "hi", "q1", "FC"], ["qwen3-8b", "E045", "en", "hi", "q1", "RM-S2"], ["qwen3-8b", "E002", "hi", "hg", "q1", "OR-S1"], ["llama3.1-8b", "P037", "hg", "hg", "q2", "RM-S3"], ["llama3.1-8b", "F024", "hi", "en", "q2", "RM-S1"], ["qwen3-8b", "E036", "hi", "hg", "q1", "OR-S2"], ["llama3.1-8b", "P015", "hi", "hi", "q2", "RM-S3"], ["qwen3-8b", "P023", "en", "en", "q2", "RM-S2"], ["qwen3-8b", "P022", "hg", "hi", "q1", "OR-S2"], ["llama3.1-8b", "F014", "en", "hi", "q2", "RM-S1"], ["llama3.1-8b", "P019", "hi", "hg", "q2", "RM-S3"], ["llama3.1-8b", "E002", "hi", "hg", "q1", "OR-S1"], ["llama3.1-8b", "E010", "hg", "hg", "q1", "FC"], ["llama3.1-8b", "E033", "hg", "hi", "q1", "RM-S2"], ["llama3.1-8b", "E028", "hg", "hi", "q2", "OR-S3"], ["qwen3-8b", "P026", "hi", "en", "q1", "RM-S1"], ["qwen3-8b", "F026", "hi", "hi", "q2", "OR-S3"], ["qwen3-8b", "F017", "hi", "en", "q2", "OR-S2"], ["qwen3-8b", "E028", "hg", "hg", "q1", "OR-S3"], ["llama3.1-8b", "P042", "hg", "hi", "q1", "FC"], ["llama3.1-8b", "P030", "en", "en", "q2", "RM-S1"], ["llama3.1-8b", "P023", "hi", "en", "q1", "OR-S1"], ["llama3.1-8b", "F018", "en", "hg", "q2", "RM-S1"], ["llama3.1-8b", "P035", "hg", "hg", "q1", "RM-S2"], ["qwen3-8b", "F033", "hi", "hg", "q1", "RM-S2"], ["llama3.1-8b", "P047", "hg", "hi", "q2", "OR-S2"], ["qwen3-8b", "F026", "hi", "hg", "q2", "RM-S1"], ["llama3.1-8b", "F034", "en", "en", "q2", "RM-S2"], ["llama3.1-8b", "F026", "hi", "hg", "q1", "RM-S2"], ["qwen3-8b", "F022", "hg", "en", "q1", "OR-S2"], ["qwen3-8b", "F047", "en", "hg", "q2", "OR-S2"], ["llama3.1-8b", "E028", "hg", "en", "q2", "OR-S1"], ["llama3.1-8b", "F002", "en", "hg", "q1", "OR-S2"], ["qwen3-8b", "F016", "en", "en", "q1", "RM-S1"], ["qwen3-8b", "F001", "hi", "hi", "q1", "OR-S1"], ["qwen3-8b", "E034", "hg", "en", "q1", "RM-S1"], ["llama3.1-8b", "P003", "hg", "hg", "q2", "RM-S1"], ["qwen3-8b", "P045", "en", "hi", "q2", "OR-S3"], ["qwen3-8b", "F007", "hg", "hi", "q2", "OR-S3"], ["llama3.1-8b", "E009", "hi", "en", "q2", "RM-S2"], ["llama3.1-8b", "P028", "hg", "hi", "q2", "RM-S3"], ["llama3.1-8b", "P039", "hi", "hi", "q1", "RM-S2"], ["qwen3-8b", "F027", "en", "en", "q1", "OR-S2"], ["qwen3-8b", "E029", "hg", "en", "q1", "OR-S1"], ["qwen3-8b", "E031", "hg", "en", "q1", "FC"], ["qwen3-8b", "F012", "hg", "hg", "q2", "RM-S2"], ["llama3.1-8b", "F027", "hg", "en", "q2", "OR-S3"], ["qwen3-8b", "F040", "hi", "en", "q1", "OR-S3"], ["qwen3-8b", "E015", "en", "hg", "q2", "OR-S1"], ["llama3.1-8b", "E044", "hg", "hi", "q2", "RM-S1"], ["llama3.1-8b", "E015", "en", "en", "q1", "FC"], ["qwen3-8b", "E006", "hi", "hg", "q2", "RM-S3"], ["llama3.1-8b", "F005", "hi", "hi", "q2", "OR-S1"], ["qwen3-8b", "E008", "hi", "hg", "q2", "OR-S3"], ["llama3.1-8b", "E024", "hg", "en", "q2", "RM-S1"], ["qwen3-8b", "E037", "hg", "en", "q1", "RM-S3"], ["qwen3-8b", "F040", "en", "en", "q1", "OR-S1"], ["qwen3-8b", "P040", "hg", "hi", "q1", "OR-S1"], ["llama3.1-8b", "F031", "hi", "hi", "q2", "OR-S3"], ["llama3.1-8b", "F031", "en", "en", "q2", "OR-S3"], ["qwen3-8b", "E009", "en", "hi", "q1", "FC"], ["llama3.1-8b", "E037", "hg", "hg", "q2", "OR-S3"], ["llama3.1-8b", "F033", "hg", "en", "q2", "RM-S3"], ["llama3.1-8b", "E019", "hi", "hg", "q2", "OR-S3"], ["llama3.1-8b", "E048", "en", "en", "q1", "OR-S2"], ["llama3.1-8b", "E021", "en", "en", "q2", "OR-S1"], ["qwen3-8b", "P037", "hg", "hi", "q2", "RM-S3"], ["llama3.1-8b", "F050", "en", "en", "q2", "RM-S3"], ["qwen3-8b", "F027", "hg", "hi", "q2", "RM-S2"], ["qwen3-8b", "P002", "en", "en", "q2", "OR-S3"], ["qwen3-8b", "E002", "hi", "hi", "q1", "RM-S2"], ["qwen3-8b", "F022", "hi", "en", "q2", "FC"], ["llama3.1-8b", "E017", "hg", "hg", "q1", "OR-S1"], ["qwen3-8b", "E032", "hi", "hi", "q2", "FC"], ["llama3.1-8b", "E015", "en", "hg", "q2", "FC"], ["llama3.1-8b", "P045", "en", "hg", "q1", "OR-S1"], ["qwen3-8b", "P028", "hg", "hg", "q1", "FC"], ["llama3.1-8b", "E012", "hg", "en", "q1", "OR-S2"], ["llama3.1-8b", "F006", "hi", "hg", "q2", "FC"], ["llama3.1-8b", "E048", "en", "hi", "q2", "OR-S1"], ["qwen3-8b", "F048", "hi", "hi", "q2", "RM-S1"], ["qwen3-8b", "F023", "hg", "hg", "q2", "RM-S3"], ["qwen3-8b", "P022", "hi", "en", "q2", "RM-S2"], ["llama3.1-8b", "E017", "hi", "hi", "q2", "FC"], ["qwen3-8b", "P050", "en", "hg", "q2", "RM-S1"], ["llama3.1-8b", "F050", "en", "hi", "q2", "OR-S2"], ["qwen3-8b", "F041", "en", "hg", "q2", "OR-S3"], ["llama3.1-8b", "E006", "hg", "hg", "q1", "OR-S2"], ["llama3.1-8b", "P014", "hg", "en", "q2", "RM-S2"], ["llama3.1-8b", "P008", "hi", "hi", "q2", "RM-S1"], ["llama3.1-8b", "E045", "hi", "hg", "q2", "RM-S1"], ["qwen3-8b", "E046", "hg", "hi", "q2", "RM-S1"], ["qwen3-8b", "P035", "en", "hg", "q2", "RM-S3"], ["qwen3-8b", "P023", "en", "hi", "q2", "RM-S3"], ["qwen3-8b", "E042", "hi", "hi", "q1", "OR-S2"], ["llama3.1-8b", "E031", "hi", "en", "q1", "OR-S2"], ["qwen3-8b", "F019", "hg", "hg", "q2", "RM-S1"], ["qwen3-8b", "P009", "en", "hi", "q1", "RM-S1"], ["qwen3-8b", "F028", "hg", "en", "q2", "OR-S1"], ["llama3.1-8b", "P033", "hi", "hi", "q1", "FC"], ["qwen3-8b", "P020", "hi", "en", "q2", "RM-S3"], ["qwen3-8b", "F041", "hg", "hg", "q2", "FC"], ["llama3.1-8b", "F029", "hi", "en", "q2", "OR-S1"], ["qwen3-8b", "P008", "hg", "hi", "q2", "RM-S3"], ["qwen3-8b", "P007", "en", "en", "q1", "OR-S2"], ["qwen3-8b", "P005", "hi", "hi", "q2", "FC"], ["llama3.1-8b", "F036", "en", "hg", "q1", "OR-S2"], ["qwen3-8b", "P020", "hi", "hg", "q2", "RM-S2"], ["llama3.1-8b", "F013", "hg", "en", "q2", "OR-S1"], ["llama3.1-8b", "F032", "hi", "hg", "q2", "RM-S3"], ["llama3.1-8b", "P008", "hg", "hg", "q2", "OR-S3"], ["qwen3-8b", "P036", "hg", "hi", "q2", "OR-S1"], ["qwen3-8b", "F013", "hg", "hi", "q2", "OR-S2"], ["qwen3-8b", "P004", "hg", "en", "q2", "RM-S2"], ["llama3.1-8b", "F030", "hg", "hg", "q2", "OR-S1"], ["qwen3-8b", "E033", "en", "hg", "q2", "OR-S3"], ["llama3.1-8b", "P033", "en", "en", "q2", "RM-S1"], ["llama3.1-8b", "F045", "hi", "en", "q2", "OR-S3"], ["llama3.1-8b", "E039", "hg", "hg", "q1", "RM-S2"], ["qwen3-8b", "E050", "hg", "hg", "q1", "OR-S1"], ["llama3.1-8b", "P014", "en", "hi", "q2", "OR-S1"], ["qwen3-8b", "P035", "hi", "en", "q1", "RM-S1"], ["llama3.1-8b", "F029", "hi", "hi", "q2", "OR-S3"], ["qwen3-8b", "P004", "en", "en", "q2", "RM-S2"], ["qwen3-8b", "P036", "hg", "hi", "q2", "RM-S1"], ["llama3.1-8b", "F016", "en", "hg", "q2", "RM-S1"], ["qwen3-8b", "F030", "hi", "hi", "q2", "OR-S2"], ["qwen3-8b", "P044", "hg", "hi", "q2", "FC"], ["qwen3-8b", "P003", "en", "hg", "q2", "RM-S1"], ["qwen3-8b", "F036", "hi", "hg", "q2", "OR-S1"], ["llama3.1-8b", "F030", "hg", "hg", "q1", "RM-S1"], ["llama3.1-8b", "F038", "hi", "hg", "q1", "FC"], ["llama3.1-8b", "F013", "hg", "hi", "q1", "RM-S3"], ["qwen3-8b", "E028", "en", "hi", "q1", "OR-S2"], ["llama3.1-8b", "P006", "hi", "en", "q1", "FC"], ["llama3.1-8b", "P009", "en", "hi", "q2", "RM-S1"], ["llama3.1-8b", "P008", "hg", "hg", "q2", "OR-S2"], ["qwen3-8b", "P040", "hi", "en", "q1", "FC"], ["llama3.1-8b", "P033", "hi", "en", "q1", "RM-S3"], ["qwen3-8b", "F013", "hg", "en", "q1", "OR-S2"], ["qwen3-8b", "P019", "en", "hi", "q1", "RM-S1"], ["qwen3-8b", "F013", "en", "hg", "q1", "OR-S2"], ["llama3.1-8b", "E044", "en", "hi", "q1", "FC"], ["qwen3-8b", "P035", "hi", "hi", "q2", "RM-S1"], ["llama3.1-8b", "F048", "hi", "hg", "q1", "RM-S1"], ["llama3.1-8b", "P035", "hi", "hi", "q1", "RM-S3"], ["qwen3-8b", "F005", "en", "hi", "q1", "OR-S3"], ["qwen3-8b", "F006", "hi", "hg", "q1", "OR-S3"], ["qwen3-8b", "F030", "hi", "en", "q1", "OR-S3"], ["llama3.1-8b", "F046", "hi", "hg", "q1", "RM-S2"], ["qwen3-8b", "F030", "hi", "en", "q2", "OR-S1"], ["qwen3-8b", "P033", "en", "hi", "q1", "RM-S2"], ["llama3.1-8b", "P008", "hi", "hg", "q2", "OR-S3"], ["llama3.1-8b", "F030", "hi", "hi", "q2", "RM-S2"], ["qwen3-8b", "F013", "hi", "hi", "q2", "RM-S2"], ["qwen3-8b", "F006", "hi", "hi", "q1", "RM-S3"], ["qwen3-8b", "F048", "en", "hg", "q2", "RM-S3"], ["qwen3-8b", "P033", "en", "en", "q2", "RM-S1"], ["qwen3-8b", "E050", "en", "hg", "q1", "OR-S1"], ["qwen3-8b", "F013", "hg", "en", "q1", "OR-S3"], ["llama3.1-8b", "P050", "hg", "hi", "q1", "RM-S1"], ["llama3.1-8b", "E001", "en", "hg", "q1", "OR-S2"], ["qwen3-8b", "F006", "hg", "en", "q1", "OR-S1"], ["llama3.1-8b", "E009", "hg", "hg", "q2", "RM-S2"], ["qwen3-8b", "F014", "hg", "hg", "q2", "FC"], ["llama3.1-8b", "E039", "hi", "hi", "q2", "OR-S2"], ["llama3.1-8b", "F016", "en", "en", "q2", "RM-S2"], ["qwen3-8b", "P003", "en", "en", "q2", "OR-S3"], ["llama3.1-8b", "E013", "hi", "hg", "q2", "OR-S2"], ["qwen3-8b", "F006", "en", "hg", "q1", "RM-S3"], ["qwen3-8b", "P003", "en", "en", "q2", "RM-S2"], ["qwen3-8b", "P014", "hg", "hi", "q1", "FC"], ["llama3.1-8b", "P020", "hg", "en", "q2", "FC"], ["llama3.1-8b", "E013", "hi", "en", "q2", "RM-S2"], ["qwen3-8b", "P004", "hi", "hi", "q2", "RM-S3"], ["qwen3-8b", "F006", "hg", "en", "q1", "RM-S2"], ["qwen3-8b", "F045", "hi", "hg", "q1", "OR-S3"], ["llama3.1-8b", "E009", "hi", "hi", "q2", "RM-S3"], ["qwen3-8b", "E013", "hi", "hg", "q2", "OR-S2"], ["llama3.1-8b", "P033", "hg", "hg", "q2", "RM-S3"], ["llama3.1-8b", "P003", "en", "en", "q2", "RM-S3"], ["qwen3-8b", "P004", "en", "hg", "q1", "RM-S1"], ["llama3.1-8b", "F020", "en", "hg", "q2", "OR-S3"], ["qwen3-8b", "P004", "hi", "hi", "q2", "RM-S1"], ["llama3.1-8b", "E009", "hg", "en", "q2", "RM-S3"], ["qwen3-8b", "E009", "hg", "hg", "q2", "RM-S1"], ["qwen3-8b", "F017", "en", "hg", "q1", "OR-S1"], ["qwen3-8b", "P003", "hi", "en", "q2", "OR-S2"], ["llama3.1-8b", "F015", "en", "hg", "q1", "FC"], ["qwen3-8b", "P004", "hg", "hi", "q2", "RM-S1"], ["qwen3-8b", "E043", "en", "hg", "q1", "OR-S3"], ["llama3.1-8b", "P033", "en", "hg", "q1", "RM-S1"], ["qwen3-8b", "P036", "hi", "en", "q1", "FC"], ["llama3.1-8b", "P004", "en", "hi", "q2", "RM-S2"], ["llama3.1-8b", "P033", "en", "hg", "q1", "RM-S3"], ["llama3.1-8b", "E009", "hg", "hg", "q2", "FC"], ["qwen3-8b", "F006", "hi", "en", "q1", "RM-S2"], ["qwen3-8b", "P003", "hi", "en", "q2", "RM-S3"], ["llama3.1-8b", "E013", "hi", "en", "q2", "OR-S2"], ["llama3.1-8b", "P035", "hi", "hg", "q2", "OR-S1"], ["qwen3-8b", "P035", "hi", "hg", "q1", "OR-S1"], ["llama3.1-8b", "F037", "en", "hi", "q2", "FC"], ["qwen3-8b", "F045", "hg", "hi", "q2", "RM-S2"], ["llama3.1-8b", "F035", "hg", "en", "q1", "RM-S2"], ["llama3.1-8b", "E020", "hi", "en", "q2", "OR-S1"], ["qwen3-8b", "P003", "en", "en", "q2", "OR-S1"], ["llama3.1-8b", "E013", "hi", "hg", "q2", "OR-S3"], ["qwen3-8b", "P035", "hi", "hg", "q2", "RM-S1"], ["llama3.1-8b", "E009", "hi", "hg", "q2", "RM-S3"], ["qwen3-8b", "E024", "en", "hg", "q1", "FC"], ["llama3.1-8b", "E039", "hg", "hi", "q2", "OR-S2"], ["qwen3-8b", "P003", "hi", "en", "q2", "RM-S1"], ["llama3.1-8b", "P004", "hg", "hi", "q2", "RM-S2"], ["qwen3-8b", "E043", "en", "hg", "q1", "OR-S2"], ["llama3.1-8b", "P033", "en", "hi", "q2", "RM-S3"], ["llama3.1-8b", "F001", "hg", "hg", "q2", "RM-S1"], ["llama3.1-8b", "F035", "hg", "en", "q1", "OR-S2"], ["llama3.1-8b", "P035", "hi", "hi", "q2", "OR-S1"], ["llama3.1-8b", "E042", "hg", "hg", "q2", "OR-S2"], ["qwen3-8b", "P035", "hi", "hi", "q1", "OR-S1"], ["qwen3-8b", "P004", "hg", "en", "q1", "FC"], ["qwen3-8b", "F006", "hg", "hg", "q1", "RM-S3"], ["qwen3-8b", "F030", "en", "hi", "q2", "FC"], ["qwen3-8b", "P003", "en", "en", "q2", "RM-S3"], ["qwen3-8b", "F028-ABS", "hi", "hg", "q1", "OR-S2"], ["llama3.1-8b", "F001-ABS", "hg", "en", "q1", "FC"], ["llama3.1-8b", "E032-ABS", "hi", "hi", "q2", "OR-S1"], ["qwen3-8b", "E032-ABS", "en", "hg", "q1", "RM-S1"], ["llama3.1-8b", "F030-ABS", "en", "en", "q2", "RM-S3"], ["qwen3-8b", "F045-ABS", "hi", "en", "q1", "OR-S1"], ["qwen3-8b", "E040-ABS", "hi", "hi", "q2", "OR-S1"], ["llama3.1-8b", "F030-ABS", "en", "hi", "q1", "RM-S3"], ["llama3.1-8b", "F001-ABS", "hi", "en", "q1", "RM-S3"], ["llama3.1-8b", "E001-ABS", "en", "hg", "q2", "FC"], ["llama3.1-8b", "P019-ABS", "hg", "hg", "q1", "FC"], ["qwen3-8b", "E001-ABS", "hi", "hg", "q2", "RM-S3"], ["llama3.1-8b", "E032-ABS", "en", "hg", "q1", "OR-S1"], ["qwen3-8b", "P022-ABS", "hg", "hg", "q1", "RM-S3"], ["llama3.1-8b", "E032-ABS", "hg", "hi", "q2", "OR-S2"], ["llama3.1-8b", "E001-ABS", "hi", "hi", "q2", "RM-S1"], ["llama3.1-8b", "F019-ABS", "hi", "hg", "q2", "FC"], ["qwen3-8b", "E001-ABS", "hg", "hg", "q2", "OR-S1"], ["llama3.1-8b", "F030-ABS", "en", "en", "q2", "OR-S2"], ["llama3.1-8b", "E001-ABS", "hg", "hi", "q1", "OR-S1"], ["llama3.1-8b", "P022-ABS", "hg", "hg", "q1", "RM-S1"], ["qwen3-8b", "P003-ABS", "hg", "hg", "q1", "OR-S2"], ["qwen3-8b", "E001-ABS", "hg", "hg", "q1", "RM-S1"], ["qwen3-8b", "E001-ABS", "en", "hg", "q2", "RM-S3"], ["llama3.1-8b", "F019-ABS", "en", "hg", "q2", "RM-S1"], ["qwen3-8b", "F019-ABS", "hi", "hg", "q1", "RM-S1"], ["qwen3-8b", "E046-ABS", "hg", "hi", "q2", "OR-S2"], ["qwen3-8b", "E001-ABS", "en", "en", "q1", "RM-S2"], ["llama3.1-8b", "E032-ABS", "en", "hi", "q2", "OR-S2"], ["llama3.1-8b", "P003-ABS", "hg", "hi", "q1", "RM-S1"], ["qwen3-8b", "F001-ABS", "en", "hi", "q1", "OR-S3"], ["qwen3-8b", "P019-ABS", "hi", "hg", "q1", "OR-S2"], ["llama3.1-8b", "E021-ABS", "hi", "en", "q1", "OR-S1"], ["llama3.1-8b", "F045-ABS", "en", "en", "q1", "OR-S3"], ["qwen3-8b", "P039-ABS", "en", "hi", "q2", "FC"], ["llama3.1-8b", "P026-ABS", "hg", "hi", "q1", "FC"], ["qwen3-8b", "F019-ABS", "hi", "hi", "q1", "RM-S3"], ["llama3.1-8b", "F001-ABS", "en", "hi", "q2", "FC"], ["qwen3-8b", "P022-ABS", "en", "hg", "q1", "OR-S1"], ["llama3.1-8b", "E040-ABS", "hi", "hi", "q1", "RM-S2"], ["qwen3-8b", "F001-ABS", "hi", "hg", "q2", "RM-S3"], ["llama3.1-8b", "E021-ABS", "hi", "hg", "q1", "RM-S3"], ["qwen3-8b", "E046-ABS", "hi", "hi", "q1", "OR-S3"], ["llama3.1-8b", "P039-ABS", "en", "hg", "q2", "OR-S1"], ["llama3.1-8b", "E046-ABS", "en", "hi", "q2", "RM-S2"], ["llama3.1-8b", "P003-ABS", "hi", "en", "q1", "FC"], ["qwen3-8b", "P026-ABS", "hg", "hi", "q1", "RM-S3"], ["qwen3-8b", "P039-ABS", "hi", "en", "q2", "RM-S1"]]


In [ ]:
%%writefile /kaggle/working/verify/run_aya.py
"""Run Aya Expanse 8B on both verification sets (greedy, fp16, 2xT4)."""
import json, time
from vllm import LLM, SamplingParams

llm = LLM(model="CohereLabs/aya-expanse-8b", dtype="half", tensor_parallel_size=2, max_model_len=4096,
          gpu_memory_utilization=0.88, seed=0)

def run(inp, out, max_tokens):
    rows = [json.loads(l) for l in open(inp, encoding="utf-8")]
    t = time.time()
    res = llm.chat([r["messages"] for r in rows], SamplingParams(temperature=0.0, max_tokens=max_tokens), use_tqdm=True)
    with open(out, "w", encoding="utf-8") as f:
        for r, o in zip(rows, res):
            r = {k: v for k, v in r.items() if k != "messages"}
            r["aya_raw"] = o.outputs[0].text.strip()
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{out}: {len(rows)} rows in {time.time() - t:.0f}s")

run("/kaggle/working/verify/judge_sample.jsonl", "/kaggle/working/verify/aya_judge.jsonl", 16)
run("/kaggle/working/verify/item_audit.jsonl", "/kaggle/working/verify/aya_audit.jsonl", 300)


In [ ]:
hits = glob.glob("/kaggle/input/**/clm/src/clm/prompts.py", recursive=True)
if not hits:
    raise RuntimeError("STOP: add the output of clm-main-r1-4-judge as input (Add Input -> Your Work)")
CLM = hits[0].rsplit("/src/clm/prompts.py", 1)[0]
print("clm root:", CLM)
sys.path[:0] = [CLM + "/src", "/kaggle/working/verify"]
import pandas as pd
from clm import prompts
import verify_build as vb
items = {j["item_id"]: j for j in map(json.loads, open(CLM + "/data/items/items_main.jsonl", encoding="utf-8"))}
scored = pd.read_csv(CLM + "/results/main_r1/scored.csv")
keys = json.load(open("/kaggle/working/verify/judge_keys.json"))
J = vb.dumps(vb.judge_rows(keys, scored, items, prompts))
A = vb.dumps(vb.audit_rows(items))
for name, text, want in (("judge_sample", J, "f0f2a279a13468abe93f2ff961d88310e2ccd2725cbb27fc701161245f645932"), ("item_audit", A, "598e199bf0c3351b0f6963d71359507416b197b0e3aca24ecf29462afbf41764")):
    got = hashlib.sha256(text.encode("utf-8")).hexdigest()
    print(name, text.count("\n"), "rows, sha256 match:", got == want)
    if got != want:
        raise RuntimeError(f"STOP: {name} differs from the locally built file")
    open(f"/kaggle/working/verify/{name}.jsonl", "w", encoding="utf-8").write(text)

## 3. Run Aya (separate process so the GPUs are released cleanly)

In [ ]:
r = subprocess.run(["python", "/kaggle/working/verify/run_aya.py"], capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-5000:]); raise RuntimeError("STOP: Aya run failed (see above)")

## 4. Quick look and save

In [ ]:
for f in ("aya_judge", "aya_audit"):
    rows = [json.loads(l) for l in open(f"/kaggle/working/verify/{f}.jsonl", encoding="utf-8")]
    print(f, len(rows))
    for x in rows[:3]:
        print("  ", x["aya_raw"][:160].replace("\n", " "))
shutil.make_archive("/kaggle/working/verify_aya_results", "zip", "/kaggle/working/verify")
print("Download zip: /kaggle/working/verify_aya_results.zip (Output tab)")